In [1]:
import pandas as pd
import numpy as np
from feature_eng import preprocess
from logging_helper import log
from sklearn.model_selection import train_test_split
df = pd.read_csv("~/ML/Data/house_prices/train.csv")
x_train,x_test = train_test_split(df,test_size=0.2,random_state=42)

x_train,x_test,y_train,y_test = preprocess(x_train,x_test)


/home/ldavi/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.model_selection import KFold, GridSearchCV

kf = KFold(n_splits=5, shuffle=True, random_state=42)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Lasso(max_iter=10000)),
])

param_grid = {
    'scaler': [StandardScaler(), MinMaxScaler(), RobustScaler()],
    'model__alpha': [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1],
}

grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    return_train_score=True,
    n_jobs=-1,
)

In [8]:
from feature_eng import correlation_filter
x_corr = correlation_filter(x_train,y_train,0.7)
result = grid_search.fit(x_corr,y_train)

In [9]:
print(grid_search.best_params_)
print(-grid_search.best_score_)  # negate to get actual RMSE

{'model__alpha': 0.0005, 'scaler': MinMaxScaler()}
0.1383209283932038


In [12]:
log(x_corr,y_train,x_test[x_corr.columns],y_test,result,"Lasso Regression with correlation")


Initialized MLflow to track repo "ldavi22/cs229-231"

Repository ldavi22/cs229-231 initialized!

🏃 View run Lasso Regression with correlation at: https://dagshub.com/ldavi22/cs229-231.mlflow/#/experiments/1/runs/9b9b601b019f4bd39f68e8fccc850b6b
🧪 View experiment at: https://dagshub.com/ldavi22/cs229-231.mlflow/#/experiments/1
